# 03 — DCIR detection, review, and measurement
This notebook asks: *does the elaborated synthetic record contain a pulse candidate, and what resistance is obtained after explicit review?* It uses a compact rest/pulse/rest pattern and runs in seconds. Current is in amperes, voltage in volts, and resistance in ohms; positive current is charge. Detection produces a Pending candidate — this notebook visibly changes it to Accepted only for the documented synthetic example.


In [ ]:
import tempfile
from pathlib import Path


def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples" / "data" / "synthetic_cycling.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the gregorithm repository folder.")


PROJECT_ROOT = find_project_root()
LOCAL_FILE = None  # Optional: Path(r"D:/your-local/canonical-file.csv")
SOURCE = (
    Path(LOCAL_FILE)
    if LOCAL_FILE is not None
    else PROJECT_ROOT / "examples" / "data" / "synthetic_cycling.csv"
)
_session = tempfile.TemporaryDirectory(prefix="gregorithm-notebook-")
WORK = Path(_session.name)
print(f"Input: {SOURCE.name}")
print(f"Temporary result area: {WORK}")

In [ ]:
from gregorithm import ElaborationRequest, ImportRequest, elaborate, import_source

imported = import_source(
    ImportRequest(SOURCE, WORK / "import", "synthetic-cell", "import-job", "import-v1")
)
elaborated = elaborate(
    ElaborationRequest(
        imported.ref.path / "points" / "canonical.parquet",
        imported.report.content_hash,
        WORK / "elaboration",
        "synthetic-cell",
        "elaboration-job",
        "elaboration-v1",
    )
)
print(f"Retained rows: {imported.report.retained_rows}")
print(f"Cycles: {elaborated.cycles}; half-cycles: {elaborated.half_cycles}")

In [ ]:
from gregorithm import DcirDetectionParameters, DcirDetectionRequest, PulsePolarity, detect_dcir

detection_parameters = DcirDetectionParameters(
    pulse_s=3.0,
    pulse_tolerance_pct=0.0,
    rest_before_s=3.0,
    rest_before_tolerance_pct=0.0,
    rest_after_s=3.0,
    rest_after_tolerance_pct=0.0,
    polarity=PulsePolarity.DISCHARGE,
)
detected = detect_dcir(
    DcirDetectionRequest(
        elaborated.ref.path,
        elaborated.fingerprint,
        WORK / "dcir-detection",
        "synthetic-cell",
        "dcir-detection-job",
        "dcir-detection-v1",
        detection_parameters,
    )
)
print(f"Pending candidates: {detected.events}")

In [ ]:
import json

records = [
    json.loads(line)
    for line in (detected.ref.path / "tables" / "dcir_events.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
]
records

The compact parameters exactly describe this teaching fixture; they are not universal pulse criteria. Select duration, tolerances, polarity, and rest requirements for the experimental protocol and review each candidate.


In [ ]:
from gregorithm import DcirEvent, EventSource, ReviewState, review_event

record = records[0]
pending = DcirEvent(
    **{
        **record,
        "source": EventSource(record["source"]),
        "state": ReviewState(record["state"]),
        "polarity": PulsePolarity(record["polarity"]) if record["polarity"] else None,
    }
)
print(f"Before review: {pending.state.value}")
accepted = review_event(pending, ReviewState.ACCEPTED, acceptance_order=0)
print(f"After explicit review: {accepted.state.value}")

In [ ]:
from gregorithm import measure_automatic_from_generation

measurement = measure_automatic_from_generation(
    elaborated.ref.path, elaborated.fingerprint, accepted, requested_time_s=2.0, label="DCIR1"
)
print(f"Reference: I={measurement.i_ref_a:.3f} A, V={measurement.v_ref_v:.3f} V")
print(f"Pulse point: I={measurement.i_meas_a:.3f} A, V={measurement.v_meas_v:.3f} V")
print(
    f"delta I={measurement.delta_i_a:.3f} A; |delta V/delta I|={measurement.resistance_ohm:.6f} ohm"
)
print(f"Actual measurement time: {measurement.actual_time_s:.1f} s")
print(f"Diagnostics: {[item.code.value for item in measurement.diagnostics]}")

In [ ]:
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

points = pq.read_table(elaborated.ref.path / "points" / "elaborated.parquet")
rows = points.to_pylist()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r["time_s"] for r in rows], [r["voltage_v"] for r in rows], marker="o", label="Voltage")
ax.axvspan(
    accepted.event_time_s,
    accepted.event_time_s + accepted.pulse_duration_s,
    alpha=0.2,
    label="Detected pulse",
)
ax.scatter(
    [measurement.reference_sample.time_s], [measurement.v_ref_v], label="Reference", zorder=3
)
ax.scatter(
    [measurement.voltage_sample.time_s], [measurement.v_meas_v], label="Measurement", zorder=3
)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (V)")
ax.set_title("Synthetic DCIR pulse")
ax.legend()
fig

Resistance uses real recorded samples and the absolute ratio |delta V/delta I|. Acceptance is a scientific decision, not an automatic consequence of detection. Report requested and actual times, diagnostics, pulse criteria, and accepted-event provenance.
